In [1]:
import sys
print(sys.version)

3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]


In [2]:
# --- Python 3.11 env + Kernel ---
!sudo apt-get install -y python3.11 python3.11-venv -qq
!python3.11 -m venv /content/py311env
!/content/py311env/bin/pip install -q --upgrade pip
!/content/py311env/bin/pip install -q ipykernel
!/content/py311env/bin/python -m ipykernel install --user --name py311 --display-name "Python 3.11 (Colab)"

# --- Projekt-Pakete ---
!/content/py311env/bin/pip install -q numpy scanpy scib pooch gdown matplotlib
!/content/py311env/bin/pip install -q torch
!/content/py311env/bin/pip install -q scgpt

Installed kernelspec py311 in /root/.local/share/jupyter/kernels/py311


In [3]:
import os
import torch
import scgpt as scg
import scgpt.tasks.cell_emb as cell_emb_mod
from pathlib import Path
import pooch
import gdown
import scanpy as sc


ModuleNotFoundError: No module named 'scgpt'

In [7]:
sys.path.insert(0, "../")

model_dir = Path("./scGPT_CP")

In [12]:
# Target directory
model_dir = "./scGPT_human"
os.makedirs(model_dir, exist_ok=True)

# Direct file IDs extracted from your links
file_ids = {
    "args.json": "15TEZmd2cZCrHwgfE424fgQkGUZCXiYrR",
    "best_model.pt": "1x1SfmFdI-zcocmqWAd7ZTC9CTEAVfKZq",
    "vocab.json": "1jfT_T5n8WNbO9QZcLWObLdRG8lYFKH-Q",
}

for filename, file_id in file_ids.items():
    output_path = os.path.join(model_dir, filename)

    if filename == "best_model.pt":
        # Clean up corrupted HTML file if it was previously downloaded
        if os.path.exists(output_path) and os.path.getsize(output_path) < 1024 * 1024:
            print(f"Removing corrupted or incomplete {filename}...")
            os.remove(output_path)

        # Check if the valid weights file is already present
        if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
            print(f"Skipping {filename} (already exists).")
            continue

        print(f"Downloading {filename} with gdown...")
        url = f"https://drive.google.com/uc?id={file_id}"
        gdown.download(url=url, output=output_path, quiet=False)

    else:
        # Pooch natively manages caching and checks if the file exists
        print(f"Downloading/fetching {filename} with pooch...")
        url = f"https://drive.google.com/uc?export=download&id={file_id}"
        pooch.retrieve(
            url=url,
            known_hash=None,
            path=model_dir,
            fname=filename,
            progressbar=True,
        )

print(f"\nFinished! Files are located in {os.path.abspath(model_dir)}")

Downloading/fetching args.json with pooch...


Downloading...
From (original): https://drive.google.com/uc?id=1x1SfmFdI-zcocmqWAd7ZTC9CTEAVfKZq
From (redirected): https://drive.google.com/uc?id=1x1SfmFdI-zcocmqWAd7ZTC9CTEAVfKZq&confirm=t&uuid=29a9a359-d45c-4887-b63a-98e4ea87817d
To: /content/scGPT_human/best_model.pt
100%|██████████| 208M/208M [00:03<00:00, 64.3MB/s] 


Downloading/fetching vocab.json with pooch...


  0%|                                              | 0.00/1.32M [00:00<?, ?B/s]

SHA256 hash of downloaded file: ee2b2c90158eedb97c2318e49abaaed0a02c6fdf7e3f7ca6a821906413c4d2a4
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.



Finished! Files are located in /content/scGPT_human


In [16]:
"""
Calculate the metrics for integration results
"""
def scib_eval(adata, batch_key, cell_type_key, embed_key):
    results = scib.metrics.metrics(
        adata,
        adata_int=adata,
        batch_key=batch_key,
        label_key=cell_type_key,
        embed=embed_key,
        isolated_labels_asw_=False,
        silhouette_=True,
        hvg_score_=False,
        graph_conn_=True,
        pcr_=True,
        isolated_labels_f1_=False,
        trajectory_=False,
        nmi_=True,  # use the clustering, bias to the best matching
        ari_=True,  # use the clustering, bias to the best matching
        cell_cycle_=False,
        kBET_=False,  # kBET return nan sometimes, need to examine
        ilisi_=False,
        clisi_=False,
    )
    result_dict = results[0].to_dict()
    
    # compute avgBIO metrics
    result_dict["avg_bio"] = np.mean(
        [
            result_dict["NMI_cluster/label"],
            result_dict["ARI_cluster/label"],
            result_dict["ASW_label"],
        ]
    )
    
    # compute avgBATCH metrics
    result_dict["avg_batch"] = np.mean(
        [
            result_dict["graph_conn"],
            result_dict["ASW_label/batch"],
        ]
    )
    
    result_dict = {k: v for k, v in result_dict.items() if not np.isnan(v)}
    
    return result_dict

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [25]:
print(os.listdir('/content/drive/MyDrive/Colab Notebooks'))

['adata_integrated.h5ad', 'scgpt.ipynb']


In [26]:
sample_data_path = '/content/drive/MyDrive/Colab Notebooks/adata_integrated.h5ad'
adata = sc.read_h5ad(sample_data_path)

gene_col = "Gene Symbol"
cell_type_key = "celltype"
batch_key = "tech"
N_HVG = 2000

In [27]:
adata.var[gene_col] = adata.var.index.values

In [28]:
org_adata = adata.copy()

In [29]:
# preprocess
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
# highly variable genes
# If raw counts are in a layer (e.g., 'counts'):
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=N_HVG,
    flavor="seurat_v3",
    layer="counts"  # Points directly to the raw counts
)
adata = adata[:, adata.var['highly_variable']]

/tmp/ipykernel_3631/372355635.py:2: UserWarning: Some cells have zero counts
  sc.pp.normalize_total(adata, target_sum=1e4)
/usr/local/lib/python3.13/dist-packages/scanpy/preprocessing/_simple.py:376: RuntimeWarning: invalid value encountered in log1p
  np.log1p(x, out=x)


In [34]:
import sys
import types

class MinimalCppVocab:
    def __init__(self, ordered_dict, min_freq=1):
        self.stoi = {}
        self.itos = []
        self.default_index = -1
        
        for word, freq in ordered_dict.items():
            if freq >= min_freq and word not in self.stoi:
                self.stoi[word] = len(self.itos)
                self.itos.append(word)

    def __getitem__(self, token):
        if token in self.stoi:
            return self.stoi[token]
        if self.default_index != -1:
            return self.default_index
        raise KeyError(f"Token '{token}' not found.")

    def __len__(self):
        return len(self.itos)

    def __contains__(self, token):
        return token in self.stoi

    def insert_token(self, token, index):
        if token not in self.stoi:
            self.itos.insert(index, token)
            self.stoi = {tok: i for i, tok in enumerate(self.itos)}

    def append_token(self, token):
        if token not in self.stoi:
            self.stoi[token] = len(self.itos)
            self.itos.append(token)

    def lookup_token(self, index):
        return self.itos[index]

    def lookup_indices(self, tokens):
        return [self[t] for t in tokens]

    def lookup_tokens(self, indices):
        return [self.itos[i] for i in indices]

    def set_default_index(self, index):
        self.default_index = index

    def get_default_index(self):
        return self.default_index


class MinimalVocabWrapper:
    def __init__(self, ordered_dict, min_freq=1):
        # scGPT accesses _vocab.vocab directly in super().__init__
        self.vocab = MinimalCppVocab(ordered_dict, min_freq=min_freq)

    def __getattr__(self, name):
        return getattr(self.vocab, name)


# Set up the mock torchtext modules
try:
    import torchtext
    import torchtext.vocab
except ImportError:
    torchtext = types.ModuleType("torchtext")
    torchtext.vocab = types.ModuleType("torchtext.vocab")
    sys.modules["torchtext"] = torchtext
    sys.modules["torchtext.vocab"] = torchtext.vocab

# Provide Vocab class base and factory function
torchtext.vocab.Vocab = MinimalCppVocab
torchtext.vocab.vocab = lambda ordered_dict, min_freq=1: MinimalVocabWrapper(ordered_dict, min_freq=min_freq)

In [35]:
import os
import torch
import scgpt as scg
import scgpt.tasks.cell_emb as cell_emb_mod

# 1. macOS affinity patch
os.sched_getaffinity = lambda _: {0}

# 2. Patch DataLoader workers
_orig_loader = cell_emb_mod.DataLoader
def _forced_zero_workers(*args, **kwargs):
    kwargs["num_workers"] = 0
    return _orig_loader(*args, **kwargs)
cell_emb_mod.DataLoader = _forced_zero_workers

# 3. Run on CPU (Notice device="cpu" passed directly to embed_data)
embed_adata = scg.tasks.embed_data(
    adata,
    model_dir,
    gene_col=gene_col,
    device="gpu",               # <-- Pass explicitly here
    batch_size=32,              # 32 or 64 is recommended for CPU inference
    use_fast_transformer=False,
)

AttributeError: 'MinimalCppVocab' object has no attribute 'copy'